# Воркшоп №4 — Evaluation

**Курсовая:** «Использование мультимодальных эмбеддингов в рекомендательных системах».

Сравниваю 7 методов на двух сценариях:

- **Warm** — трек есть в CF-графе
- **Cold** — новый трек, CF-сигнала нет

Метрики: Recall@10, nDCG@10, MRR. Ground truth — top-10 соседей из `edges.parquet` (Last.fm getSimilar).

Методы: CF-baseline, TEXT/IMAGE/AUDIO-only, Early/Late/Cross-attention fusion.

In [1]:
%pip install -q numpy pandas tqdm pyarrow torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("eval")

K = 10
TEST_FRAC = 0.20       # доля warm pool в test
COLD_FRAC = 0.15       # доля всех src = cold (без CF-сигнала)
SEED = 42

DATA_DIR = Path("data")
EMB_DIR = DATA_DIR / "embeddings"
FUSION_DIR = DATA_DIR / "fusion"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)

## 1. Загрузка данных

In [3]:
def l2norm(x: np.ndarray) -> np.ndarray:
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


# Эмбеддинги
text_npz = np.load(EMB_DIR / "text.npz")
image_npz = np.load(EMB_DIR / "image.npz")
audio_npz = np.load(EMB_DIR / "audio.npz")
early_npz = np.load(FUSION_DIR / "early_fusion.npz")
cross_npz = np.load(FUSION_DIR / "cross_attention.npz")
late_cfg = json.loads((FUSION_DIR / "late_fusion_weights.json").read_text())

TRACK_IDS = text_npz["track_ids"].astype(np.int64)
N = len(TRACK_IDS)
tid_to_idx = {int(t): i for i, t in enumerate(TRACK_IDS)}

text_emb = l2norm(text_npz["emb"].astype(np.float32))
image_emb = l2norm(image_npz["emb"].astype(np.float32))
audio_emb = l2norm(audio_npz["emb"].astype(np.float32))
early_emb = l2norm(early_npz["emb"].astype(np.float32))
cross_emb = l2norm(cross_npz["emb"].astype(np.float32))
late_w = tuple(late_cfg["weights"])

# Ground truth + train-граф
edges_df = pd.read_parquet(DATA_DIR / "edges.parquet")
edges_in = edges_df.loc[edges_df["in_corpus"] & edges_df["dst_track_id"].notna()].copy()
edges_in["dst_track_id"] = edges_in["dst_track_id"].astype(np.int64)
edges_in = edges_in.loc[
    edges_in["src_track_id"].isin(tid_to_idx) & edges_in["dst_track_id"].isin(tid_to_idx)
]

gt_by_src: dict[int, list[tuple[int, float]]] = {}
for src_tid, grp in edges_in.sort_values("match", ascending=False).groupby("src_track_id"):
    pairs = [(int(r.dst_track_id), float(r.match)) for r in grp.head(K).itertuples()]
    if pairs:
        gt_by_src[int(src_tid)] = pairs

all_src = sorted(gt_by_src.keys())
log.info("Tracks=%d, src with GT=%d", N, len(all_src))

2026-06-10 02:50:34,527 | INFO | Tracks=14787, src with GT=7606


## 2. Train / Test / Cold split

15% src-треков — cold (убираю из CF-графа). 20% оставшихся — warm test.

In [4]:
# Cold items: 15% src-треков полностью удалены из CF-графа (модель «нового трека»)
n_cold = max(1, int(len(all_src) * COLD_FRAC))
cold_src = set(rng.choice(all_src, size=n_cold, replace=False))

# Warm pool: остальные треки (у них есть коллаборативный сигнал)
warm_pool = [s for s in all_src if s not in cold_src]
rng.shuffle(warm_pool)

n_test = max(1, int(len(warm_pool) * TEST_FRAC))
warm_test_src = set(warm_pool[:n_test])
train_warm_src = set(warm_pool[n_test:])

# CF-граф: все рёбра, где src НЕ в cold_src
train_edges = edges_in.loc[~edges_in["src_track_id"].isin(cold_src)]
cf_graph: dict[int, list[tuple[int, float]]] = {}
for src_tid, grp in train_edges.sort_values("match", ascending=False).groupby("src_track_id"):
    cf_graph[int(src_tid)] = [
        (int(r.dst_track_id), float(r.match)) for r in grp.itertuples()
    ]

print(f"Cold queries (no CF signal): {len(cold_src)}")
print(f"Warm test queries:           {len(warm_test_src)}")
print(f"Warm train queries:          {len(train_warm_src)}")
print(f"CF graph src coverage:       {len(cf_graph)}")
print(f"Warm test with CF edges:     {sum(1 for s in warm_test_src if s in cf_graph)} / {len(warm_test_src)}")

Cold queries (no CF signal): 1140
Warm test queries:           1293
Warm train queries:          5173
CF graph src coverage:       6466
Warm test with CF edges:     1293 / 1293


## 3. Метрики и функции рекомендации

In [5]:
def _gt_set(gt_pairs: list[tuple[int, float]]) -> set[int]:
    return {d for d, _ in gt_pairs}


def _gt_rel(gt_pairs: list[tuple[int, float]]) -> dict[int, float]:
    return {d: m for d, m in gt_pairs}


def recall_at_k(pred: list[int], gt: set[int], k: int = K) -> float:
    if not gt:
        return 0.0
    return len(set(pred[:k]) & gt) / len(gt)


def ndcg_at_k(pred: list[int], gt_rel: dict[int, float], k: int = K) -> float:
    if not gt_rel:
        return 0.0
    dcg = 0.0
    for i, p in enumerate(pred[:k]):
        if p in gt_rel:
            dcg += gt_rel[p] / np.log2(i + 2)
    ideal = sorted(gt_rel.values(), reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def mrr(pred: list[int], gt: set[int]) -> float:
    for i, p in enumerate(pred):
        if p in gt:
            return 1.0 / (i + 1)
    return 0.0


def cf_recommend(src_tid: int, k: int = K) -> list[int]:
    pairs = cf_graph.get(src_tid, [])
    pairs = sorted(pairs, key=lambda x: -x[1])
    return [d for d, _ in pairs[:k]]


def knn_recommend_vector(src_idx: int, emb: np.ndarray, k: int = K) -> list[int]:
    sims = emb[src_idx] @ emb.T
    sims[src_idx] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    top = top[np.argsort(-sims[top])]
    return [int(TRACK_IDS[i]) for i in top]


def knn_recommend_late(src_idx: int, weights: tuple[float, float, float], k: int = K) -> list[int]:
    w_t, w_i, w_a = weights
    sims = (
        w_t * (text_emb[src_idx] @ text_emb.T)
        + w_i * (image_emb[src_idx] @ image_emb.T)
        + w_a * (audio_emb[src_idx] @ audio_emb.T)
    )
    sims[src_idx] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    top = top[np.argsort(-sims[top])]
    return [int(TRACK_IDS[i]) for i in top]


def evaluate_method(
    name: str,
    queries: set[int],
    recommend_fn,
) -> dict[str, float]:
    recalls, ndcgs, mrrs = [], [], []
    for src_tid in tqdm(queries, desc=name, leave=False):
        if src_tid not in gt_by_src:
            continue
        gt_pairs = gt_by_src[src_tid]
        gt = _gt_set(gt_pairs)
        gt_rel = _gt_rel(gt_pairs)
        pred = recommend_fn(src_tid)
        recalls.append(recall_at_k(pred, gt))
        ndcgs.append(ndcg_at_k(pred, gt_rel))
        mrrs.append(mrr(pred, gt))
    return {
        "method": name,
        "recall@10": float(np.mean(recalls)) if recalls else 0.0,
        "ndcg@10": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "mrr": float(np.mean(mrrs)) if mrrs else 0.0,
        "n_queries": len(recalls),
    }

## 4. Warm-сценарий

In [6]:
METHODS = [
    ("CF-baseline", lambda tid: cf_recommend(tid)),
    ("TEXT-only", lambda tid: knn_recommend_vector(tid_to_idx[tid], text_emb)),
    ("IMAGE-only", lambda tid: knn_recommend_vector(tid_to_idx[tid], image_emb)),
    ("AUDIO-only", lambda tid: knn_recommend_vector(tid_to_idx[tid], audio_emb)),
    ("Early fusion", lambda tid: knn_recommend_vector(tid_to_idx[tid], early_emb)),
    ("Late fusion", lambda tid: knn_recommend_late(tid_to_idx[tid], late_w)),
    ("Cross-attention", lambda tid: knn_recommend_vector(tid_to_idx[tid], cross_emb)),
]

warm_rows = []
t0 = time.time()
for name, fn in METHODS:
    row = evaluate_method(name, warm_test_src, fn)
    row["scenario"] = "warm"
    warm_rows.append(row)
    print(f"[warm] {name:16s}  R@10={row['recall@10']:.3f}  nDCG@10={row['ndcg@10']:.3f}  MRR={row['mrr']:.3f}")

warm_df = pd.DataFrame(warm_rows)
log.info("Warm eval done in %.1fs", time.time() - t0)
warm_df

CF-baseline:   0%|          | 0/1293 [00:00<?, ?it/s]

[warm] CF-baseline       R@10=1.000  nDCG@10=1.000  MRR=1.000


TEXT-only:   0%|          | 0/1293 [00:00<?, ?it/s]

[warm] TEXT-only         R@10=0.082  nDCG@10=0.119  MRR=0.172


IMAGE-only:   0%|          | 0/1293 [00:00<?, ?it/s]

...

[warm] Late fusion       R@10=0.113  nDCG@10=0.152  MRR=0.219


Cross-attention:   0%|          | 0/1293 [00:00<?, ?it/s]

2026-06-10 02:50:45,543 | INFO | Warm eval done in 9.3s


[warm] Cross-attention   R@10=0.115  nDCG@10=0.133  MRR=0.204


,method,recall@10,ndcg@10,mrr,n_queries,scenario
0,CF-baseline,1.000000,1.000000,1.000000,1293,warm
1,TEXT-only,0.082389,0.118501,0.172418,1293,warm
2,IMAGE-only,0.081404,0.125405,0.188140,1293,warm
3,AUDIO-only,0.022656,0.023760,0.045401,1293,warm
4,Early fusion,0.102928,0.146539,0.217354,1293,warm
5,Late fusion,0.112920,0.152095,0.218824,1293,warm
6,Cross-attention,0.114888,0.133434,0.203746,1293,warm


## 5. Cold-сценарий

Главный результат курсовой: CF падает, контентные методы работают.

In [7]:
cold_rows = []
t0 = time.time()
for name, fn in METHODS:
    row = evaluate_method(name, cold_src, fn)
    row["scenario"] = "cold"
    cold_rows.append(row)
    print(f"[cold] {name:16s}  R@10={row['recall@10']:.3f}  nDCG@10={row['ndcg@10']:.3f}  MRR={row['mrr']:.3f}")

cold_df = pd.DataFrame(cold_rows)
log.info("Cold eval done in %.1fs", time.time() - t0)
cold_df

CF-baseline:   0%|          | 0/1140 [00:00<?, ?it/s]

[cold] CF-baseline       R@10=0.000  nDCG@10=0.000  MRR=0.000


TEXT-only:   0%|          | 0/1140 [00:00<?, ?it/s]

[cold] TEXT-only         R@10=0.082  nDCG@10=0.126  MRR=0.187


IMAGE-only:   0%|          | 0/1140 [00:00<?, ?it/s]

...

[cold] Late fusion       R@10=0.110  nDCG@10=0.159  MRR=0.236


Cross-attention:   0%|          | 0/1140 [00:00<?, ?it/s]

2026-06-10 02:50:53,734 | INFO | Cold eval done in 8.2s


[cold] Cross-attention   R@10=0.110  nDCG@10=0.135  MRR=0.212


,method,recall@10,ndcg@10,mrr,n_queries,scenario
0,CF-baseline,0.000000,0.000000,0.000000,1140,cold
1,TEXT-only,0.082029,0.125656,0.186849,1140,cold
2,IMAGE-only,0.080163,0.128308,0.190997,1140,cold
3,AUDIO-only,0.023035,0.022419,0.044466,1140,cold
4,Early fusion,0.100033,0.150521,0.228887,1140,cold
5,Late fusion,0.110062,0.159104,0.236173,1140,cold
6,Cross-attention,0.110024,0.135304,0.211544,1140,cold


## 6. Сводная таблица

In [8]:
final_df = pd.concat([warm_df, cold_df], ignore_index=True)
final_df[["recall@10", "ndcg@10", "mrr"]] = final_df[["recall@10", "ndcg@10", "mrr"]].round(3)

pivot = final_df.pivot(index="method", columns="scenario", values="recall@10")
pivot["cold_vs_text"] = (pivot["cold"] / pivot.get("TEXT-only", pivot["cold"])).round(2) if "TEXT-only" in pivot.index else np.nan

print("=== Recall@10: warm vs cold ===")
print(pivot.sort_values("cold", ascending=False).to_string())
print()

OUT_PATH = RESULTS_DIR / "eval_results.csv"
final_df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

# Ablation: multimodal vs text-only на cold
text_cold = final_df.loc[(final_df.method == "TEXT-only") & (final_df.scenario == "cold"), "recall@10"].iloc[0]
print("\n=== Cold-start ablation (Recall@10) ===")
for m in ["TEXT-only", "Early fusion", "Late fusion", "Cross-attention"]:
    v = final_df.loc[(final_df.method == m) & (final_df.scenario == "cold"), "recall@10"].iloc[0]
    delta = (v - text_cold) / max(text_cold, 1e-9) * 100
    print(f"  {m:18s} {v:.3f}  ({delta:+.0f}% vs text-only)")

=== Recall@10: warm vs cold ===
scenario          cold   warm  cold_vs_text
method                                     
Cross-attention  0.110  0.115           1.0
Late fusion      0.110  0.113           1.0
Early fusion     0.100  0.103           1.0
TEXT-only        0.082  0.082           1.0
IMAGE-only       0.080  0.081           1.0
AUDIO-only       0.023  0.023           1.0
CF-baseline      0.000  1.000           NaN

Saved: data/results/eval_results.csv

=== Cold-start ablation (Recall@10) ===
  TEXT-only          0.082  (+0% vs text-only)
  Early fusion       0.100  (+22% vs text-only)
  Late fusion        0.110  (+34% vs text-only)
  Cross-attention    0.110  (+34% vs text-only)


## 7. Выводы

На cold CF = 0, мультимодальные методы ≈ 0.11. Результаты в `data/results/eval_results.csv`.